# DAY 1

In [42]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

In [43]:
# The files have no header row, so we name the columns ourselves
train = pd.read_csv("train.csv", header=None, names=["label", "text"])
test_pool = pd.read_csv("test.csv", header=None, names=["label", "text"])

print("Train rows:", len(train))
print("Test pool rows:", len(test_pool))

Train rows: 560000
Test pool rows: 38000


In [44]:
# In this dataset, 1 = negative and 2 = positive
label_names = {1: "negative", 2: "positive"}

train["y"] = train["label"].map(label_names)
test_pool["y"] = test_pool["label"].map(label_names)

print(train[["label", "y"]].drop_duplicates())

   label         y
0      1  negative
1      2  positive


In [45]:
# Keep 300 reviews from the test file, split evenly between classes.
# The first output (everything else) is not needed, so it goes to "_".
_, test_300 = train_test_split(
    test_pool,
    test_size=300,
    stratify=test_pool["y"],
    random_state=42,
)

test_300.to_csv("test_300.csv", index=False)

print("Test set size:", len(test_300))
print(test_300["y"].value_counts())

Test set size: 300
y
negative    150
positive    150
Name: count, dtype: int64


In [46]:
# 1. Class balance
print("Class balance:")
print(train["y"].value_counts(normalize=True))

# 2. Average review length
print("\nAverage length:", round(train["text"].str.len().mean()), "characters")

# 3. Missing text
print("Missing texts:", train["text"].isna().sum())

# 4. Ten random examples per class
for name in ["negative", "positive"]:
    print(f"\n--- 10 random {name} examples ---")
    examples = train[train["y"] == name].sample(10, random_state=1)
    for text in examples["text"]:
        print("-", text[:150])

Class balance:
y
negative    0.5
positive    0.5
Name: proportion, dtype: float64

Average length: 726 characters
Missing texts: 0

--- 10 random negative examples ---
- Wow. Talk about disappointing!! We tried to go once, but the line was ridiculous, so we tried a different day hoping that it would be mind blowing. It
- I recently stayed at the MGM Grand and was surprised at how dated and worn the rooms were.\n\nThere were scuffs on the walls and blisters on the furni
- The staff is incompetent, I will never go here again. The sushi is mediocre at best -- there are much better Japanese restaurants in Madison, stay cle
- If you've been to Olive Garden, you've been here and you probably paid less for the same food and better service. \n\nThe only reason I can see people
- My husband and I went on a tour over a month ago.  The front desk staff is rather icy. Our tour guide was nice however.  We got a week guest pass in t
- I have never written a negative review and feel badly for doing s

In [47]:
hard_texts = [
    "The food was not bad at all.",                      # negation
    "Oh great, another hour-long wait for cold fries.",  # sarcasm
    "Lovely staff, but the pasta was a disaster.",       # mixed
    "I wouldn’t say the service was good.",
    "The soup was not bad at all.",
    "Terrible parking, but the food was worth every minute.",
    "Wow, I love paying $30 for a lukewarm burger.",
    "Absolutely the best burger in town!",
    "The steak was alright but I had better.",
    "I found the texture of the food to tough.",
    "I wish the service was better.",
    "The selection of the menu was limited.",
    "The preperation time for the food took a while.",
    "The taste of the food was supprisingly satisfying.",
    "They provided grate value for the prices of their food.",
    "The allowed for special orders to accomidate my needs.",
    "Their selection of beverages was severly limited.",
    "I found both the service and the food as average",
    "Everything that they provided was expetional!",
    "The provided incredible customer service!",
    "The atmosphere didn't match the food at all",
    "They didn't provide any options for milder flavors of noodles",
    "The pasta came out a little overcooked in my opinion",
    "The seafood selection was didn't have a non-shellfish option",
    "The buffet had a large selection but the quality wasn't better than average",
    "The quality of food and services was worth the high price point",
    "I would highly recommend visting for both the food and ambiance at least once",
    "Their approach was both unique and a must visit if you're in the area",
    "Their barbeque is some of the best i have eaten while on vacation",
    "The service was almose immediate with how fast everything was prepared."
]

hard_labels = [
    "positive",
    "negative",
    "negative",
    "negative",
    "positive",
    "positive",
    "negative",
    "positive",
    "positive",
    "negative",
    "negative",
    "negative",
    "negative",
    "positive",
    "positive",
    "positive",
    "negative",
    "negative",
    "positive",
    "posotive",
    "negative",
    "negative",
    "negative",
    "negative",
    "negative",
    "positive",
    "positive",
    "positive",
    "positive",
    "positive"
]

print(len(hard_texts), len(hard_labels))
assert len(hard_texts) == len(hard_labels), "Texts and labels don't match!"

hard = pd.DataFrame({"text": hard_texts, "y": hard_labels})
hard.to_csv("hard_cases.csv", index=False)

30 30


## EDA Notes

- Dataset: Yelp polarity reviews (train.csv / test.csv), binary sentiment.
  Train has 560,000 reviews and the test file has 38,000 reviews (from len() in my Day 1 output).
- Labels: 1 = negative, 2 = positive. No neutral class.
  I added a readable column "y" (negative/positive) and kept the original label column.
- Class balance: train is 50/50. test_300 is 150 negative / 150 positive.
- Average review length: about 726 characters, so reviews are long.
- Missing text: 0.
- Observations from reading 10 random reviews per class (train only):
  - They weren't always focused on the food, some also placed emphesis on the location.
  - Some included personal experinces tha influnced their review.
  - Service of the restauraunt was seen to contribute along side the food.
- Possible label noise: some reviews look mixed or opposite to their label
  (e.g. one positive-labeled review described dry, flavorless food).
- Test set: 300 examples drawn from test.csv with train_test_split
  (stratified, random_state=42), saved as test_300.csv. All models are scored on this
  same file. I did not look at it while building anything.
- Hard cases: 30 sentences I wrote myself (not taken from the dataset), labeled
  positive or negative before running any model. They cover negation, sarcasm,
  mixed sentiment, and understatement. Saved as hard_cases.csv.

In [48]:
import vaderSentiment, nltk, anthropic, sklearn

print("API key found:", os.environ.get("ANTHROPIC_API_KEY") is not None)
print("All packages imported")

API key found: True
All packages imported
